# E14 — Quem cai primeiro

O capítulo anterior mediu a direção do tempo dentro de **uma** série. A última pergunta da
travessia é a mesma, no par: quando as coisas ruins vêm juntas, **alguém cai primeiro**, e essa
ordem se repete?

O instrumento é uma contagem. Um episódio começa no dia em que uma das pernas rompe o próprio
corte, desde que nenhum episódio tenha começado nos dias anteriores; se a outra perna romper dentro da mesma janela seguinte, o
episódio é dirigido e tem um líder. O que se mede é a diferença entre os episódios liderados por
uma perna e os liderados pela outra, dividida pelo total.

A regra tem um viés próprio e o caderno o declara antes de usar: a espera olha para trás e a
detecção olha para a frente, de modo que num par sem adiantamento nenhum os dois lados já não
saem iguais. Por isso o número do par real só vale contra o número dos pares sorteados, e nunca
contra zero. Pelo mesmo motivo a inversão do relógio não confere este instrumento: virar o mundo
ao contrário não vira a regra.


In [1]:
# <- brinque com: SERIE_A, SERIE_B, JANELA, JANELAS_EPISODIO, ATRASOS, NULOS, SEMENTES
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, dependencia, graficos, mudanca, volatilidade

SERIE_A = "sp500.csv"
SERIE_B = "ibov.csv"
JANELA = 252
CAUDA = 0.05
JANELAS_EPISODIO = (1, 5, 10, 21)
EPISODIO = 5
ATRASOS = (0, 1, 2, 5, 10, 21)
NULOS = 40
SEMENTES = 300
PEDACOS = 4

retornos_a = volatilidade.retornos_log(dados.carregar_serie(SERIE_A))
retornos_b = volatilidade.retornos_log(dados.carregar_serie(SERIE_B))
comuns = retornos_a.index.intersection(retornos_b.index)
rompe_a = dependencia.rompimentos(retornos_a, JANELA, CAUDA)
rompe_b = dependencia.rompimentos(retornos_b, JANELA, CAUDA)
junto = dependencia.juntos(rompe_a, rompe_b)
print("%s | %d dias comuns | rompe A %d | rompe B %d | juntos %d | %s vezes a independencia"
      % (SERIE_A, junto["dias"], junto["rompe_a"], junto["rompe_b"], junto["juntos"],
         ("%.2f" % junto["excesso"]).replace(".", ",")))
print("acompanhados: %s dos rompimentos de A vem com B no mesmo dia"
      % ("%.3f" % junto["acompanhados"]).replace(".", ","))


sp500.csv | 6204 dias comuns | rompe A 315 | rompe B 316 | juntos 115 | 7,17 vezes a independencia
acompanhados: 0,365 dos rompimentos de A vem com B no mesmo dia


In [2]:
# O par real: quem lidera, em cada janela de episodio.
linhas = []
for janela in JANELAS_EPISODIO:
    e = dependencia.episodios_dirigidos(rompe_a, rompe_b, janela)
    linhas.append({"janela": janela, **{k: e[k] for k in ("lider_a", "lider_b", "sozinho_a",
                                                          "sozinho_b", "juntos", "assimetria")}})
tabela = pd.DataFrame(linhas).set_index("janela")
print(tabela.round(3).to_string())
principal = dependencia.episodios_dirigidos(rompe_a, rompe_b, EPISODIO)
print()
print("na janela de %d dias: A lidera %d, B lidera %d, assimetria %+.3f"
      % (EPISODIO, principal["lider_a"], principal["lider_b"], principal["assimetria"]))


        lider_a  lider_b  sozinho_a  sozinho_b  juntos  assimetria
janela                                                            
1            17        5        160        171      99       0.545
5            35       20         84        108      61       0.273
10           40       24         50         76      44       0.250
21           48       18         20         42      29       0.455

na janela de 5 dias: A lidera 35, B lidera 20, assimetria +0.273


In [3]:
# O nulo: pares sorteados, contemporaneos, medidos com a mesma regra.
assimetrias_nulas = []
for i in range(NULOS):
    x, y = mudanca.dependencia(comuns.size, np.random.default_rng(SEMENTES + i), 0.012, 0.5, 0.5,
                               quando=comuns.size // 2)
    px = dependencia.rompimentos(pd.Series(x, index=comuns))
    py = dependencia.rompimentos(pd.Series(y, index=comuns))
    assimetrias_nulas.append(dependencia.episodios_dirigidos(px, py, EPISODIO)["assimetria"])
assimetrias_nulas = np.array(assimetrias_nulas)
media_nula = float(assimetrias_nulas.mean())
desvio_nulo = float(assimetrias_nulas.std(ddof=1))
print("nulo: media %+.3f | dispersao %.3f | maior em modulo %+.3f | %d pares"
      % (media_nula, desvio_nulo, assimetrias_nulas[np.argmax(np.abs(assimetrias_nulas))], NULOS))
print("o par real, contra o nulo: %.2f desvios" % ((principal["assimetria"] - media_nula) / desvio_nulo))


nulo: media -0.014 | dispersao 0.107 | maior em modulo +0.242 | 40 pares
o par real, contra o nulo: 2.68 desvios


In [4]:
# O controle: um adiantamento declarado, posto no par pelo pareamento.
linhas = []
for atraso in ATRASOS:
    b_deslocado = dependencia.pareado(retornos_b, atraso).dropna()
    index = comuns.intersection(b_deslocado.index)
    ra = dependencia.rompimentos(retornos_a).reindex(index).fillna(False).astype(bool)
    rb = dependencia.rompimentos(b_deslocado).reindex(index).fillna(False).astype(bool)
    e = dependencia.episodios_dirigidos(ra, rb, EPISODIO)
    linhas.append({"atraso": atraso, "lider_a": e["lider_a"], "lider_b": e["lider_b"],
                   "assimetria": e["assimetria"]})
controle = pd.DataFrame(linhas).set_index("atraso")
print(controle.round(3).to_string())
print()
print("com %d dias de atraso declarado a assimetria cai a %+.3f, e o nulo esta em %+.3f"
      % (ATRASOS[-1], controle.loc[ATRASOS[-1], "assimetria"], media_nula))


        lider_a  lider_b  assimetria
atraso                              
0            35       20       0.273
1            87       25       0.554
2            80       22       0.569
5            71       35       0.340
10           39       36       0.040
21           32       26       0.103

com 21 dias de atraso declarado a assimetria cai a +0.103, e o nulo esta em -0.014


In [5]:
# A estabilidade: o mesmo par, partido em pedacos.
pedacos = []
for i, pedaco in enumerate(np.array_split(np.arange(comuns.size), PEDACOS), start=1):
    index = comuns[pedaco]
    e = dependencia.episodios_dirigidos(rompe_a.reindex(index), rompe_b.reindex(index), EPISODIO)
    pedacos.append({"pedaco": i, "dias": int(index.size), **{k: e[k] for k in
                   ("lider_a", "lider_b", "dirigidos", "assimetria")}})
pedacos = pd.DataFrame(pedacos).set_index("pedaco")
print(pedacos.round(3).to_string())
print()
print("pedacos acima da media do nulo: %d de %d" % (int((pedacos["assimetria"] > media_nula).sum()), PEDACOS))


        dias  lider_a  lider_b  dirigidos  assimetria
pedaco                                               
1       1613        8        6         14       0.143
2       1613       10        4         14       0.429
3       1612       10        6         16       0.250
4       1612        7        4         11       0.273

pedacos acima da media do nulo: 4 de 4


In [6]:
# Figura 1: a assimetria contra o adiantamento declarado.
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
eixo.plot(controle.index, controle["assimetria"], marker="o", color="#1f4e79", lw=1.6,
          label="com adiantamento declarado")
eixo.axhline(principal["assimetria"], color="#b03a2e", ls="--", lw=1.4,
             label="o par como veio (%+.3f)" % principal["assimetria"])
eixo.axhspan(media_nula - 2 * desvio_nulo, media_nula + 2 * desvio_nulo, color="#555555", alpha=0.18,
             label="o nulo, dois desvios")
eixo.axhline(media_nula, color="#555555", ls=":", lw=1.2)
eixo.set_xlabel("dias de atraso declarado na segunda perna")
eixo.set_ylabel("assimetria dos episódios")
eixo.legend(frameon=False, fontsize=8)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E14_direcao_conjunta", 1)
plt.close(fig)
print("no atraso zero a assimetria e %+.3f | no maior atraso %+.3f"
      % (controle.loc[0, "assimetria"], controle.loc[ATRASOS[-1], "assimetria"]))


no atraso zero a assimetria e +0.273 | no maior atraso +0.103


In [7]:
# Figura 2: os pedacos do par contra o nulo.
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
posicoes = np.arange(PEDACOS)
eixo.bar(posicoes, pedacos["assimetria"], 0.55,
         color=["#1f4e79" if v > media_nula else "#7f8c8d" for v in pedacos["assimetria"]])
eixo.axhspan(media_nula - 2 * desvio_nulo, media_nula + 2 * desvio_nulo, color="#555555", alpha=0.18,
             label="o nulo, dois desvios")
eixo.axhline(media_nula, color="#555555", ls=":", lw=1.2, label="a média do nulo")
for i, v in enumerate(pedacos["assimetria"]):
    eixo.annotate("%+.2f" % v, (i, v), textcoords="offset points", xytext=(0, 4), ha="center",
                  fontsize=9)
eixo.set_xticks(posicoes)
eixo.set_xticklabels(["pedaço %d" % p for p in pedacos.index])
eixo.set_ylabel("assimetria dos episódios")
eixo.legend(frameon=False, fontsize=8)
eixo.grid(alpha=0.25, axis="y")
fig.tight_layout()
graficos.salvar(fig, "E14_direcao_conjunta", 2)
plt.close(fig)
print("assimetrias por pedaco: %s" % [round(v, 3) for v in pedacos["assimetria"]])


assimetrias por pedaco: [0.143, 0.429, 0.25, 0.273]


## Leitura visual das figuras

Conferida com a ponte de visão do ambiente (AGENTS.md §9) na rodada de 24 de setembro de
2026: os .png deste caderno foram relidos um a um, contra o que está escrito aqui.


Figura 1 (a assimetria contra o adiantamento declarado). O eixo horizontal traz os adiantamentos postos no par à mão — zero, um, dois, cinco, dez e vinte e um dias —, e o vertical, a assimetria dos episódios, de menos 0,2 a 0,6. A linha azul começa exatamente sobre a reta vermelha tracejada, que é o par como veio: com zero dias declarados o controle reproduz o próprio dado, e é para isso que a reta serve. Dali ela sobe e faz um pico em um e dois dias, bem acima da tracejada. Depois desce sem voltar: em cinco dias ainda está acima da reta, em dez desaba para perto de zero e em vinte e um para 0,10, já dentro da faixa cinza e ainda acima da linha pontilhada do nulo. A figura tem um pico e um cruzamento, e o que ela mostra é que declarar um dia de adiantamento não enfraquece a assimetria: fortalece. O intervalo entre as duas pernas é de um ou dois dias, e é ali que a curva toca o máximo. O que o eixo engana: os seis pontos não estão igualmente espaçados, e o pico, que é o achado, cai num pedaço do eixo dez vezes mais estreito que o último intervalo; visto de longe, ele parece um tremor no começo da linha. A faixa cinza e a linha do nulo ficam ambas rentes ao zero, e o eixo reserva um quarto da altura para a região negativa que nenhuma curva visita, de modo que a queda do fim parece mais suave do que é.

Figura 2 (os pedaços do par). Quatro barras, uma por pedaço, medidas na mesma régua, com a faixa cinza do nulo e a linha pontilhada da média do nulo ao fundo. Todas apontam para o mesmo lado: nenhuma desce abaixo da média do nulo, e as quatro saem na cor dos pedaços que ficam acima dela. As alturas estão anotadas em cima de cada barra, e a ordem não é monótona — sobe, cai e sobe de novo —, de modo que o que a figura mostra é um sinal que troca de tamanho e conserva o sentido. Três das quatro barras passam por cima da borda superior da faixa; a do primeiro pedaço fica dentro dela. O que o eixo engana: a escala desce até 0,2 negativo e nenhuma barra chega perto do chão, então um terço da altura do gráfico é espaço que nada ocupa, e isso comprime as barras umas contra as outras — a maior é três vezes a menor, e na tela a diferença parece pequena. A linha pontilhada da média do nulo fica logo abaixo do zero e convida a ler zero como ausência de direção, quando a média do nulo não é zero: uma barra que parasse em zero já seria um par com o índice liderando.

In [8]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
NOMES_JANELA = {1: "um", 5: "cinco", 10: "dez", 21: "vinte_e_um"}
NOMES_ATRASO = {0: "zero", 1: "um", 2: "dois", 5: "cinco", 10: "dez", 21: "vinte_e_um"}
NOMES_PEDACO = {1: "primeiro", 2: "segundo", 3: "terceiro", 4: "quarto"}
resultado = {
    "conjunta_dias": int(junto["dias"]),
    "conjunta_rompe_a": int(junto["rompe_a"]),
    "conjunta_rompe_b": int(junto["rompe_b"]),
    "conjunta_juntos": int(junto["juntos"]),
    "conjunta_excesso": float(junto["excesso"]),
    "conjunta_acompanhados": float(junto["acompanhados"]),
    "conjunta_janela_episodio": int(EPISODIO),
    "conjunta_nulos": int(NULOS),
    "conjunta_nulo_media": float(media_nula),
    "conjunta_nulo_dispersao": float(desvio_nulo),
    "conjunta_pedacos": int(PEDACOS),
    "conjunta_desvios": float((principal["assimetria"] - media_nula) / desvio_nulo),
    "conjunta_lider_a": int(principal["lider_a"]),
    "conjunta_lider_b": int(principal["lider_b"]),
    "conjunta_assimetria": float(principal["assimetria"]),
    "conjunta_pedacos_acima": int((pedacos["assimetria"] > media_nula).sum()),
    "conjunta_atraso_maior": int(ATRASOS[-1]),
}
for janela in JANELAS_EPISODIO:
    nome = NOMES_JANELA[janela]
    resultado["conjunta_lider_a_%s" % nome] = int(tabela.loc[janela, "lider_a"])
    resultado["conjunta_lider_b_%s" % nome] = int(tabela.loc[janela, "lider_b"])
    resultado["conjunta_assimetria_%s" % nome] = float(tabela.loc[janela, "assimetria"])
for atraso in ATRASOS:
    resultado["conjunta_controle_%s" % NOMES_ATRASO[atraso]] = float(controle.loc[atraso, "assimetria"])
for pedaco in pedacos.index:
    nome = NOMES_PEDACO[pedaco]
    resultado["conjunta_%s_dias" % nome] = int(pedacos.loc[pedaco, "dias"])
    resultado["conjunta_%s_lider_a" % nome] = int(pedacos.loc[pedaco, "lider_a"])
    resultado["conjunta_%s_assimetria" % nome] = float(pedacos.loc[pedaco, "assimetria"])

caminho = Path("lab/resultados/E14_direcao_conjunta.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))


lab/resultados/E14_direcao_conjunta.json gravado | 47 grandezas
